# Gold — Star Schema de Vendas

Este notebook constrói a camada **gold** do modelo de vendas a partir da tabela silver `{CATALOG}.gold.tb_sales_enriched`.

## Objetivo

Transformar os dados enriquecidos de vendas em um **star schema** dimensional pronto para consumo analítico (dashboards, BI e exploração).

## Estrutura

- **`fact_sales`** — fato de vendas com grão de 1 linha por `sales_key` (data x marca x região x canal x pacote). Chaves substitutas (`date_key`, `brand_key`, `region_key`, `channel_key`, `package_key`) geradas via `xxhash64`, com `COALESCE` para valores nulos (`'UNKNOWN'`). Volumes negativos representam devoluções/ajustes.


## Fluxo

1. Lê o parâmetro `CATALOG` (widget) para definir o schema gold de destino.
3. Cria `fact_sales` como tabela Delta com commentário documentando o grão.
4. Popula a fato via `MERGE` (upsert por `sales_key`), registrando `_gold_updated_at` como timestamp de carga.



In [0]:
dbutils.widgets.text("CATALOG", "")

CATALOG = dbutils.widgets.get("CATALOG")
G = f"{CATALOG}.gold"
SOURCE = f"{G}.tb_sales_enriched"


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {G}.fact_sales (
    sales_key        STRING,
    date_key         INT,
    brand_key        BIGINT,
    region_key       BIGINT,
    channel_key      BIGINT,
    package_key      BIGINT,
    sales_volume     DECIMAL(18,2),
    _gold_updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Fato de vendas. Grão: 1 linha por sales_key (data x marca x região x canal x pacote). Volume negativo = devolução/ajuste.'
""")

In [0]:
spark.sql(f"""
MERGE INTO {G}.fact_sales AS t
USING (
    SELECT sales_key,
           CAST(date_format(sale_date, 'yyyyMMdd') AS INT)                      AS date_key,
           xxhash64(COALESCE(CAST(ce_brand_flvr AS STRING), 'UNKNOWN'))         AS brand_key,
           xxhash64(COALESCE(btlr_org_lvl_c_desc, 'UNKNOWN'))                   AS region_key,
           xxhash64(COALESCE(trade_chnl_desc, 'UNKNOWN'))                       AS channel_key,
           xxhash64(COALESCE(tsr_pckg_nm, 'UNKNOWN'))                           AS package_key,
           sales_volume,
           current_timestamp()                                                  AS _gold_updated_at
    FROM {SOURCE}
) AS s
ON t.sales_key = s.sales_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")